In [1]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import haversine_distances

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Load wildfire observations
fires = pd.read_csv(
    project_dir / "data/processed/fires_cleaned.csv"
)

# Load EPA station observations
pm25 = pd.read_csv(
    project_dir / "data/processed/pm25_cleaned.csv",
    dtype={
        "State Code": str,
        "County Code": str,
        "Site Num": str
    }
)

# Load daily weather
weather = pd.read_csv(
    project_dir / "data/processed/statewide_weather_daily.csv",
    parse_dates=["date"]
)

print("Fire detections:", len(fires))
print("EPA observations:", len(pm25))
print("Weather records:", len(weather))


Fire detections: 21694
EPA observations: 8548
Weather records: 8924


In [2]:

site_cols = [
    "State Code",
    "County Code",
    "Site Num"
]

pm25["station_id"] = (
    pm25["State Code"].str.zfill(2)
    + "-"
    + pm25["County Code"].str.zfill(3)
    + "-"
    + pm25["Site Num"].str.zfill(4)
)

stations = (
    pm25[
        ["station_id", "Latitude", "Longitude"]
    ]
    .drop_duplicates("station_id")
    .reset_index(drop=True)
)

print("Unique stations:", len(stations))
display(stations.head())


Unique stations: 97


,station_id,Latitude,Longitude
0,06-007-0008,39.761680,-121.840470
1,06-009-0001,38.201850,-120.680277
2,06-011-0007,39.021221,-122.281803
3,06-011-1002,39.189190,-121.998870
4,06-019-0011,36.785380,-119.773210


In [3]:

# Convert station coordinates into radians
station_lat = np.radians(
    stations["Latitude"].to_numpy()
)[:, None]

station_lon = np.radians(
    stations["Longitude"].to_numpy()
)[:, None]

# Convert fire coordinates into radians
fire_lat = np.radians(
    fires["latitude"].to_numpy()
)[None, :]

fire_lon = np.radians(
    fires["longitude"].to_numpy()
)[None, :]

# Longitude difference
delta_lon = fire_lon - station_lon

# Initial great-circle bearing
x = np.sin(delta_lon) * np.cos(fire_lat)

y = (
    np.cos(station_lat) * np.sin(fire_lat)
    -
    np.sin(station_lat)
    * np.cos(fire_lat)
    * np.cos(delta_lon)
)

bearing = (
    np.degrees(np.arctan2(x, y)) + 360
) % 360

print("Bearing matrix shape:", bearing.shape)

print(
    "Bearing range:",
    round(bearing.min(), 2),
    round(bearing.max(), 2)
)


Bearing matrix shape: (97, 21694)
Bearing range: 0.0 360.0


In [4]:

EARTH_RADIUS_KM = 6371.0088

station_coordinates = np.radians(
    stations[["Latitude", "Longitude"]].to_numpy()
)

fire_coordinates = np.radians(
    fires[["latitude", "longitude"]].to_numpy()
)

distance_matrix = (
    haversine_distances(
        station_coordinates,
        fire_coordinates
    ) * EARTH_RADIUS_KM
)

print("Distance matrix:", distance_matrix.shape)

print("Minimum distance:", distance_matrix.min())
print("Maximum distance:", distance_matrix.max())


Distance matrix: (97, 21694)
Minimum distance: 1.3121876756821325
Maximum distance: 1224.683915942899


In [5]:

fires["acq_date"] = pd.to_datetime(
    fires["acq_date"]
)

fire_time = (
    fires["acq_time"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(4)
)

# Construct actual satellite acquisition timestamp
fire_datetime = pd.to_datetime(
    fires["acq_date"].dt.strftime("%Y-%m-%d")
    + " "
    + fire_time.str[:2]
    + ":"
    + fire_time.str[2:],
    utc=True,
    errors="coerce"
)

fires["datetime_local"] = (
    fire_datetime.dt.tz_convert(
        "America/Los_Angeles"
    )
)

fires["date_local"] = (
    fires["datetime_local"]
    .dt.tz_localize(None)
    .dt.normalize()
)

print(
    "Missing timestamps:",
    fires["datetime_local"].isna().sum()
)

print(
    "Local date range:",
    fires["date_local"].min(),
    fires["date_local"].max()
)


Missing timestamps: 0
Local date range: 2025-07-01 00:00:00 2025-09-30 00:00:00


In [6]:

# Ensure daily dates use the same format
weather["date"] = pd.to_datetime(weather["date"])

fire_dates = fires["date_local"].to_numpy(
    dtype="datetime64[ns]"
)

frp_values = fires["frp"].to_numpy()

# Limit spatial influence to 100 km
distance_weight = np.where(
    distance_matrix <= 100,
    np.exp(-distance_matrix / 50),
    0.0
)

# Match daily weather to our ordered stations
weather_indexed = weather.set_index(
    ["station_id", "date"]
)

study_dates = pd.date_range(
    "2025-07-01",
    "2025-09-30"
)

transport_features = []

for date in study_dates:

    # Weather for all stations on this date
    daily_station_weather = weather_indexed.loc[
        pd.MultiIndex.from_product(
            [stations["station_id"], [date]],
            names=["station_id", "date"]
        )
    ].reset_index()

    # Circular mean wind direction
    wind_direction = (
        daily_station_weather["wind_direction_mean"]
        .to_numpy()
    )

    # Angle between fire bearing and wind-from direction
    angular_difference = np.radians(
        bearing - wind_direction[:, None]
    )

    directional_weight = np.maximum(
        0,
        np.cos(angular_difference)
    )

    # Select fire detections on this date
    date_mask = (
        fire_dates == np.datetime64(date)
    )

    weighted_frp = (
        directional_weight[:, date_mask]
        * distance_weight[:, date_mask]
        * frp_values[date_mask]
    ).sum(axis=1)

    transport_features.append(
        pd.DataFrame({
            "station_id": stations["station_id"].to_numpy(),
            "date": date,
            "upwind_frp_100km": weighted_frp
        })
    )

transport_daily = pd.concat(
    transport_features,
    ignore_index=True
)

print("Transport feature records:", len(transport_daily))

display(transport_daily.head(10))


Transport feature records: 8924


,station_id,date,upwind_frp_100km
0,06-007-0008,2025-07-01,13.448666
1,06-009-0001,2025-07-01,0.355042
2,06-011-0007,2025-07-01,2.817848
3,06-011-1002,2025-07-01,3.005606
4,06-019-0011,2025-07-01,2.893097
5,06-019-0500,2025-07-01,2.572531
6,06-019-2009,2025-07-01,3.446916
7,06-019-2016,2025-07-01,2.537497
8,06-019-5001,2025-07-01,1.561863
9,06-019-5025,2025-07-01,0.000000


In [7]:

print("Summary statistics:")
display(
    transport_daily[
        "upwind_frp_100km"
    ].describe()
)

print("\nMissing values:")
print(
    transport_daily.isna().sum()
)

print("\nHighest wind-aligned FRP observations:")
display(
    transport_daily.nlargest(
        10,
        "upwind_frp_100km"
    )
)

print("\nNegative values:")
print(
    (transport_daily["upwind_frp_100km"] < 0).sum()
)


Summary statistics:


count     8924.000000
mean        21.301573
std        233.824016
min          0.000000
25%          0.000000
50%          0.045566
75%          0.975696
max      10898.488617
Name: upwind_frp_100km, dtype: float64


Missing values:
station_id          0
date                0
upwind_frp_100km    0
dtype: int64

Highest wind-aligned FRP observations:


,station_id,date,upwind_frp_100km
3183,06-083-3001,2025-08-02,10898.488617
6707,06-027-0002,2025-09-08,5469.314792
3668,06-083-3001,2025-08-07,5367.218393
3666,06-083-2004,2025-08-07,5302.470286
6782,06-107-0009,2025-09-08,5220.980479
3181,06-083-2004,2025-08-02,4762.637786
3377,06-083-3001,2025-08-04,4605.065566
3765,06-083-3001,2025-08-08,4189.273742
3280,06-083-3001,2025-08-03,3822.385768
6711,06-027-1033,2025-09-08,3662.430945



Negative values:
0


In [8]:

output_path = (
    project_dir /
    "data/processed/transport_features_daily.csv"
)

transport_daily.to_csv(
    output_path,
    index=False
)

print("Transport-aware features saved!")
print(output_path)


Transport-aware features saved!
/Users/sumanthkamath/Documents/VS Code /EcoSentinel/data/processed/transport_features_daily.csv


In [9]:

# Load our original station-day fire features
fire_daily = pd.read_csv(
    project_dir / "data/processed/fire_pm25_combined.csv",
    parse_dates=["date"]
)

# Compare original FRP with wind-aligned FRP
validation = transport_daily.merge(
    fire_daily[
        ["station_id", "date", "frp_sum_100km"]
    ],
    on=["station_id", "date"],
    how="inner",
    validate="one_to_one"
)

# Weighted FRP must not exceed original FRP
violations = (
    validation["upwind_frp_100km"]
    > validation["frp_sum_100km"] + 1e-6
)

print("Compared records:", len(validation))
print("Violations:", violations.sum())

display(
    validation[
        [
            "station_id",
            "date",
            "frp_sum_100km",
            "upwind_frp_100km"
        ]
    ].head()
)


Compared records: 8548
Violations: 0


,station_id,date,frp_sum_100km,upwind_frp_100km
0,06-007-0008,2025-07-01,55.47,13.448666
1,06-009-0001,2025-07-01,21.27,0.355042
2,06-011-0007,2025-07-01,62.69,2.817848
3,06-011-1002,2025-07-01,55.47,3.005606
4,06-019-0011,2025-07-01,113.66,2.893097


In [10]:

output_path = (
    project_dir /
    "data/processed/transport_features_daily.csv"
)

transport_daily.to_csv(output_path, index=False)

print("Saved:", output_path)


Saved: /Users/sumanthkamath/Documents/VS Code /EcoSentinel/data/processed/transport_features_daily.csv
